In [2]:

import re

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts') as f:
    src = f.read()

DEFAULT = {'nucleare':5,'sanzioni':10,'opinione':0,'defcon':7,'risorse':5,'stabilita':5,
           'risorse_iran':5,'forze_militari_iran':5,'tecnologia_nucleare_iran':5,'stabilita_iran':5,
           'risorse_coalizione':5,'influenza_diplomatica_coalizione':5,'tecnologia_avanzata_coalizione':5,'supporto_pubblico_coalizione':5,'stabilita_coalizione':5,
           'risorse_russia':5,'influenza_militare_russia':5,'veto_onu_russia':2,'stabilita_economica_russia':5,'stabilita_russia':5,
           'risorse_cina':5,'influenza_commerciale_cina':5,'cyber_warfare_cina':5,'stabilita_rotte_cina':5,'stabilita_cina':5,
           'risorse_europa':5,'influenza_diplomatica_europa':5,'aiuti_umanitari_europa':5,'coesione_ue_europa':5,'stabilita_europa':5}
ALL_T = list(DEFAULT.keys())

card_re = re.compile(r"card_id:'([^']+)',\s*card_name:'([^']+)',\s*faction:'([^']+)'.*?effects:\{([^}]+)\}", re.DOTALL)
problems = []
for m in card_re.finditer(src):
    cid, cname, faz, eff_str = m.groups()
    for t in ALL_T:
        fn = re.search(rf'{t}\s*:\s*\(v[^)]*\)\s*=>\s*([^,}}]+)', eff_str)
        if fn:
            fn_body = fn.group(1).strip().rstrip(',')
            try:
                v = float(eval(f"(lambda v:{fn_body})(DEFAULT['{t}'])", {'DEFAULT':DEFAULT}))
                if abs(v) > 3:
                    problems.append((cid, cname, t, fn_body, v))
            except: pass

print(f"Valori fuori range (|delta|>3): {len(problems)}")
for p in problems:
    print(f"  {p[0]} '{p[1]}' [{p[2]}]: fn='{p[3]}' val={p[4]}")


Valori fuori range (|delta|>3): 13
  C034 'Sequestro Petroliera' [nucleare]: fn='v+1' val=6.0
  C035 'Asse della Resistenza' [nucleare]: fn='v+1' val=6.0
  C038 'Milizie Iraq' [nucleare]: fn='v+1' val=6.0
  C043 'Esercitazioni Navali' [nucleare]: fn='v+1' val=6.0
  C047 'Rappresaglia Missilistica' [nucleare]: fn='v+1' val=6.0
  NI03 'Chiusura Stretto di Hormuz' [nucleare]: fn='v+1' val=6.0
  NI15 'Milizie Syria Attivate' [nucleare]: fn='v+1' val=6.0
  NI19 'Rafah 2.0 - Supporto Gaza' [nucleare]: fn='v+1' val=6.0
  NC01 'Accordo AIEA Mar 2026' [risorse_coalizione]: fn='v-1' val=4.0
  NC02 'Risoluzione ONU 2891' [risorse_coalizione]: fn='v-1' val=4.0
  NC04 'Proposta JCPOA 3.0' [risorse_coalizione]: fn='v-1' val=4.0
  NC05 'Summit G7 Iran Mar 2026' [risorse_coalizione]: fn='v-1' val=4.0
  NC40 'Proposta Pace Definitiva' [risorse_coalizione]: fn='v-1' val=4.0


In [5]:

import re

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts') as f:
    src = f.read()

DEFAULT = {'nucleare':5,'sanzioni':10,'opinione':0,'defcon':7,'risorse':5,'stabilita':5,
           'risorse_iran':5,'forze_militari_iran':5,'tecnologia_nucleare_iran':5,'stabilita_iran':5,
           'risorse_coalizione':5,'influenza_diplomatica_coalizione':5,'tecnologia_avanzata_coalizione':5,'supporto_pubblico_coalizione':5,'stabilita_coalizione':5,
           'risorse_russia':5,'influenza_militare_russia':5,'veto_onu_russia':2,'stabilita_economica_russia':5,'stabilita_russia':5,
           'risorse_cina':5,'influenza_commerciale_cina':5,'cyber_warfare_cina':5,'stabilita_rotte_cina':5,'stabilita_cina':5,
           'risorse_europa':5,'influenza_diplomatica_europa':5,'aiuti_umanitari_europa':5,'coesione_ue_europa':5,'stabilita_europa':5}
ALL_T = list(DEFAULT.keys())

def clamp_fn_body(fn_body):
    def replace_num(m):
        num = float(m.group(0))
        if abs(num) > 3:
            clamped = max(-3.0, min(3.0, num))
            return str(int(clamped)) if clamped == int(clamped) else str(clamped)
        return m.group(0)
    result = re.sub(r'(?<=[=>?:+\-\s])(-?\d+(?:\.\d+)?)', replace_num, fn_body)
    return result

lines = src.split('\n')
new_lines = []
changes = 0

for line in lines:
    new_line = line
    for t in ALL_T:
        fn_m = re.search(rf'({t}:\(v[^)]*\)=>)([^,}}]+)', new_line)
        if fn_m:
            prefix = fn_m.group(1)
            fn_body = fn_m.group(2).strip().rstrip(',')
            try:
                v = float(eval(f"(lambda v:{fn_body})(DEFAULT['{t}'])", {'DEFAULT':DEFAULT}))
                if abs(v) > 3:
                    new_body = clamp_fn_body(fn_body)
                    try:
                        new_v = float(eval(f"(lambda v:{new_body})(DEFAULT['{t}'])", {'DEFAULT':DEFAULT}))
                        if abs(new_v) <= 3:
                            new_line = new_line.replace(prefix + fn_body, prefix + new_body)
                            changes += 1
                            print(f"  FIX {t}: '{fn_body}' ({v}) → '{new_body}' ({new_v})")
                        else:
                            clamped = max(-3, min(3, int(round(v))))
                            new_line = new_line.replace(prefix + fn_body, prefix + str(clamped))
                            changes += 1
                            print(f"  FIX {t}: '{fn_body}' ({v}) → '{clamped}' (fallback)")
                    except:
                        pass
            except: pass
    new_lines.append(new_line)

src = '\n'.join(new_lines)
with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts', 'w') as f:
    f.write(src)
print(f"\nTotale correzioni: {changes}")

# Verifica finale
with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts') as f:
    src2 = f.read()
remaining = []
card_re = re.compile(r"card_id:'([^']+)',\s*card_name:'([^']+)',\s*faction:'([^']+)'.*?effects:\{([^}]+)\}", re.DOTALL)
for m in card_re.finditer(src2):
    cid, cname, faz, eff_str = m.groups()
    for t in ALL_T:
        fn = re.search(rf'{t}\s*:\s*\(v[^)]*\)\s*=>\s*([^,}}]+)', eff_str)
        if fn:
            try:
                v = float(eval(f"(lambda v:{fn.group(1).strip().rstrip(',')})(DEFAULT['{t}'])", {'DEFAULT':DEFAULT}))
                if abs(v) > 3:
                    remaining.append(f"{cid} {t}: {v}")
            except: pass
print(f"Valori ancora fuori range: {len(remaining)}")
for r in remaining: print(f"  ❌ {r}")


  FIX nucleare: 'v+1' (6.0) → '3' (fallback)
  FIX nucleare: 'v+1' (6.0) → '3' (fallback)
  FIX nucleare: 'v+1' (6.0) → '3' (fallback)
  FIX nucleare: 'v+1' (6.0) → '3' (fallback)
  FIX nucleare: 'v+1' (6.0) → '3' (fallback)
  FIX nucleare: 'v+1' (6.0) → '3' (fallback)
  FIX nucleare: 'v+1' (6.0) → '3' (fallback)
  FIX nucleare: 'v+1' (6.0) → '3' (fallback)
  FIX risorse_coalizione: 'v-1' (4.0) → '3' (fallback)
  FIX risorse_coalizione: 'v-1' (4.0) → '3' (fallback)
  FIX risorse_coalizione: 'v-1' (4.0) → '3' (fallback)
  FIX risorse_coalizione: 'v-1' (4.0) → '3' (fallback)
  FIX risorse_coalizione: 'v-1' (4.0) → '3' (fallback)

Totale correzioni: 13
Valori ancora fuori range: 0
